# Libraries

In [2]:
import pandas as pd
import numpy as np
import dash
from dash import html, dcc
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import plotly.express as px
from dash import no_update
import datetime as dt
import calendar

In [3]:
# ──────────────────────────────────────────────
# HS2 CHAPTER LABELS
# ──────────────────────────────────────────────
HS2_LABELS = {
    "01": "Live Animals",
    "02": "Meat & Edible Offal",
    "03": "Fish & Seafood",
    "04": "Dairy, Eggs & Honey",
    "05": "Animal Products (Bones, Ivory, etc.)",
    "06": "Live Plants, Bulbs & Cut Flowers",
    "07": "Fresh & Frozen Vegetables",
    "08": "Fresh & Frozen Fruits",
    "09": "Coffee, Tea & Spices",
    "10": "Cereals & Grains",
    "11": "Milling Products (Flour, Starch)",
    "12": "Oil Seeds, Fodder & Industrial Plants",
    "13": "Vegetable Gums, Resins & Extracts",
    "14": "Vegetable Plaiting Materials",
    "15": "Animal & Vegetable Fats & Oils",
    "16": "Prepared Meat & Seafood",
    "17": "Sugars, Syrups & Confectionery",
    "18": "Cocoa & Chocolate Products",
    "19": "Baked Goods, Pasta & Cereals",
    "20": "Preserved Fruits & Vegetables",
    "21": "Miscellaneous Food Preparations",
    "22": "Beverages (Water, Beer, Spirits)",
    "23": "Animal Feed & Food Industry Residues",
    "24": "Tobacco & Nicotine Products",
    "25": "Salt, Cement & Construction Minerals",
    "26": "Ores & Mineral Concentrates",
    "27": "Petroleum, Fuels & Related Products",
    "28": "Inorganic Chemicals & Compounds",
    "29": "Organic Chemicals",
    "30": "Pharmaceutical Products",
    "31": "Fertilizers",
    "32": "Paints, Varnishes & Dyes",
    "33": "Cosmetics, Perfumes & Toiletries",
    "34": "Soaps, Cleaning & Polishing Prep.",
    "39": "Plastics & Plastic Articles",
    "40": "Rubber & Rubber Articles",
    "44": "Wood & Wood Articles",
    "47": "Pulp of Wood",
    "48": "Paper & Paperboard",
    "49": "Printed Books & Newspapers",
    "51": "Wool & Animal Hair Fabrics",
    "52": "Cotton & Cotton Fabrics",
    "54": "Synthetic Filaments & Fabrics",
    "55": "Synthetic & Artificial Staple Fibres",
    "61": "Knitted Apparel & Clothing",
    "62": "Woven Apparel & Clothing",
    "63": "Textile Made-Up Articles & Worn Clothing",
    "64": "Footwear",
    "68": "Stone, Cement & Abrasive Articles",
    "69": "Ceramic Products",
    "70": "Glass & Glassware",
    "71": "Precious Stones, Metals & Jewellery",
    "72": "Iron & Steel",
    "73": "Iron & Steel Articles",
    "74": "Copper & Copper Articles",
    "75": "Nickel & Nickel Articles",
    "76": "Aluminium & Aluminium Articles",
    "82": "Hand Tools & Cutting Tools",
    "83": "Metal Fittings, Mountings & Hardware",
    "84": "Industrial Machinery & Mechanical Equipment",
    "85": "Electrical Equipment & Electronics",
    "86": "Rail Transport Equipment",
    "87": "Motor Vehicles & Parts",
    "88": "Aircraft & Aerospace Equipment",
    "89": "Ships & Watercraft",
    "90": "Optical, Medical & Measuring Instruments",
    "94": "Furniture, Bedding & Lighting",
    "95": "Toys, Games & Sports Equipment",
    "96": "Miscellaneous Manufactured Articles",
    "97": "Art ,Antiques & Collectibles",
    "98": "Special Transactions (Repairs, Gifts)",
    "99": "Confidential & Low-Value Transactions"
}


# Load

In [4]:
# ── Data ───────────────────────────────────────────────────────────────────────
df = pd.read_parquet(
    "Dataset/Dataset.parquet",
    engine="pyarrow"
)
 
for col in ['Commodity', 'Province', 'Country', 'trade_type']:
    df[col] = df[col].astype('category')
 
df['Year'] = df['Period'].dt.year
df['Month'] = df['Period'].dt.month
 
df_kpi = df.groupby(
    ['Year', 'Commodity', 'Province', 'Country', 'trade_type'],
    observed=True
)['Value ($)'].sum().reset_index()

In [5]:
# ── Dropdown option lists ──────────────────────────────────────────────────────
year_options      = sorted(df['Year'].unique().tolist())
commodity_options = sorted(df['Commodity'].cat.categories.tolist())
province_options  = sorted(df['Province'].cat.categories.tolist())
country_options   = sorted(df['Country'].cat.categories.tolist())
date_range_label  = f"{df['Period'].min().strftime('%b %Y')} – {df['Period'].max().strftime('%b %Y')}"

In [6]:
def fmt_value(value):
    abs_val = abs(value)
    if abs_val >= 1_000_000_000:
        return f'${value/1_000_000_000:.1f}B'
    elif abs_val >= 1_000_000:
        return f'${value/1_000_000:.1f}M'
    elif abs_val >= 1_000:
        return f'${value/1_000:.1f}K'
    else:
        return f'${value:.1f}'

In [7]:
def build_monthly_chart(filtered_df):
    monthly = (
        filtered_df
        .groupby([filtered_df['Period'].dt.year.rename('year'),
                  filtered_df['Period'].dt.month.rename('month'),
                  'trade_type'])['Value ($)']
        .sum().reset_index()
    )
    pivot = monthly.pivot_table(
        index=['year', 'month'], columns='trade_type',
        values='Value ($)', aggfunc='sum'
    ).fillna(0).reset_index()
    pivot.columns.name = None
    pivot = pivot.rename(columns={'Export': 'exports', 'Import': 'imports'})
    pivot['balance'] = pivot['exports'] - pivot['imports']
    pivot = pivot.sort_values(['year', 'month']).reset_index(drop=True)
    pivot['label'] = pivot.apply(
        lambda r: f"{calendar.month_abbr[int(r['month'])]} {int(r['year'])}", axis=1
    )

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=pivot['label'], y=pivot['exports'],
        name='Exports', marker_color="#458098"
    ))
    fig.add_trace(go.Bar(
        x=pivot['label'], y=pivot['imports'],
        name='Imports', marker_color="#183662"
    ))
    fig.add_trace(go.Scatter(
        x=pivot['label'], y=pivot['balance'] / 1e9,
        name='Balance', line=dict(color='black', width=2),
        marker=dict(size=4), yaxis='y2'
    ))
    
    tick_vals = np.linspace(0, pivot['exports'].max(), 6)

    fig.update_layout(
        barmode='group',
        
        yaxis=dict(title='Trade Value (CAD)',
                   tickvals=tick_vals,
                   ticktext=[fmt_value(v) for v in tick_vals]
                  ),
        yaxis2=dict(title='Balance (CAD $B)', 
                    overlaying='y', 
                    side='right'
                    ),
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        template='plotly_white',
        margin=dict(t=40, b=40),
    )
    return fig

In [8]:
def build_monthly_chart2(filtered_df):
    monthly = (
        filtered_df
        .groupby([filtered_df['Period'].dt.year.rename('year'),
                  filtered_df['Period'].dt.month.rename('month'),
                  'trade_type'])['Value ($)']
        .sum().reset_index()
    )
    pivot = monthly.pivot_table(
        index=['year', 'month'], columns='trade_type',
        values='Value ($)', aggfunc='sum'
    ).fillna(0).reset_index()
    pivot.columns.name = None
    pivot = pivot.rename(columns={'Export': 'exports', 'Import': 'imports'})
    pivot['balance'] = pivot['exports'] - pivot['imports']
    pivot = pivot.sort_values(['year', 'month']).reset_index(drop=True)
    pivot['label'] = pivot.apply(
        lambda r: f"{calendar.month_abbr[int(r['month'])]} {int(r['year'])}", axis=1
    )

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=pivot['label'], y=pivot['exports'],
        name='Exports', marker_color="#65CC90"
    ))
    fig.add_trace(go.Bar(
        x=pivot['label'], y=-pivot['imports'],
        name='Imports', marker_color="#1A4731"
    ))
    fig.add_trace(go.Scatter(
        x=pivot['label'], y=pivot['balance'] / 1e9,
        name='Balance', line=dict(color='black', width=2),
        marker=dict(size=4), yaxis='y2'
    ))
    
    max_val = max(pivot['exports'].max(), pivot['imports'].max())
    max_bal = pivot['balance'].abs().max() / 1e9
    tick_vals_both = np.linspace(-max_val, max_val, 9)  # symmetric around zero

    fig.update_layout(
    barmode='overlay',                          # ← key: bars relative to zero
    yaxis=dict(
        title='Trade Value (CAD)',
        tickvals=tick_vals_both,                 # include negative ticks too
        ticktext=[fmt_value(abs(v)) for v in tick_vals_both],  # show absolute values
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
    ),
    

    yaxis2=dict(
    title='Balance (CAD $B)',
    overlaying='y',
    side='right',
    tickvals=np.linspace(-max_bal, max_bal, 9),
    ticktext=[fmt_value(abs(v) * 1e9) for v in np.linspace(-max_bal, max_bal, 9)],
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    template='plotly_white',
    margin=dict(t=40, b=40),
    )
    return fig

In [9]:
df.groupby('Commodity')['Value ($)'].sum().head(1)

/var/folders/fd/zfd4k53n2kj37g0q2y1cgjr40000gn/T/ipykernel_16465/4081516233.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Commodity')['Value ($)'].sum().head(1)


Commodity
0101.21.00 - Horses, live, pure-bred breeding    8697163.0
Name: Value ($), dtype: float64

In [15]:
number =len(set(df["Commodity"]))
print(number)

16428


In [10]:
app = dash.Dash(__name__)

# ── Style  ────────────────────────────────────────────────────────────
DARK_GREEN   = '#1A4731'
MEDIUM_GREEN = '#2D6A4F'
BLUE_ACCENT  = '#1F4E79'
BLUE_MEDIUM =  '#366A9B'
LIGHT_GRAY   = '#F5F5F5'
MID_GRAY     = '#E0E0E0'
TEXT_DARK    = '#1A1A1A'
TEXT_GRAY    = '#555555'
WHITE        = '#FFFFFF'
RED          = '#C00000'
GREEN_TREND  = "#65CC90"
 
FONT_MAIN = "'Arial', 'Times New Roman', serif"
FONT_BODY = "'Helvetica Neue', 'Helvetica', Arial, sans-serif"

KPI_STYLE_ROW = {
    'display':   'flex',
    'gap':       '12px',
    'margin':    '16px 32px',  # side margins match your header padding
    'width':     'auto',       # ← not 100%, lets the margins work
    'flexWrap':  'wrap',       # ← wraps to next line on smaller screens
}

KPI_STYLE_BOX = {
    'flex':            '1',
    'backgroundColor': 'white',
    'borderRadius':    '10px',
    'padding':         '20px',
    'textAlign':       'center',
    'boxShadow':       '0 2px 8px rgba(0,0,0,0.08)'
}
KPI_STYLE_LABEL = {
    'margin':     '0',
    'fontSize':   '18px',
    'fontWeight': 'bold',
    'color':      BLUE_MEDIUM
}
KPI_STYLE_VALUE = {
    'margin':   '4px 0 0 0',
    'fontSize': '24px',
    'color':    BLUE_ACCENT
}
STYLE_CHART_ROW = {
    'display':      'flex',
    'gap':          '20px',
    'margin': '30px',
    'width':   'auto',
    'flexwrap': 'wrap'
}
STYLE_CHART_ITEM = {
    'flex':     '1',
    'minWidth': '0'
}
STYLE_DROPDOWN_LABEL = {
    'fontWeight': 'bold',
    'color':     BLUE_MEDIUM 
}
STYLE_DROPDOWN_CHILD = {
    'flex':            '1',
    'minWidth':        '150px',   # ← minimum before wrapping on small screens
    'backgroundColor': 'white',
    'borderRadius':    '5px',
    'boxShadow':       '0 2px 8px rgba(0,0,0,0.08)',
}

STYLE_DROPDOWN_ROW = {
    'display':   'flex',
    'gap':       '12px',
    'margin':    '16px 32px',  # side margins match your header padding
    'width':     'auto',       # ← not 100%, lets the margins work
    'flexWrap':  'wrap',       # ← wraps to next line on smaller screens
}

# ── Layout ─────────────────────────────────────────────────────────────────────

app.layout = html.Div(
        style={
        'fontFamily': FONT_BODY,
        'backgroundColor': LIGHT_GRAY,
        'minHeight': '100vh',
        'margin': '0',
        'padding': '0',
    },
    children=[                              

                # ── HEADER ─────────────────────────────────────────────────────────────
        html.Div(
            style={
                'backgroundColor': DARK_GREEN,
                'padding': '24px 32px',
                'display': 'flex',
                'justifyContent': 'space-between',
                'alignItems': 'center',
            },
            children=[
                html.Div([
                    html.H1(
                        'Canada Trade',
                        style={
                            'margin': '20px',
                            'fontFamily': FONT_MAIN,
                            'fontSize': '36px',
                            'fontWeight': 'bold',
                            'color': WHITE,
                            'letterSpacing': '0.5px',
                        }
                    ),
                    html.P(
                        f'Total trade flow, balance, and year-over-year trend · {date_range_label}',
                        style={
                            'margin': '4px 2px 2px 2px',
                            'fontStyle': 'italic',
                            'fontSize': '14px',
                            'color': '#B2DFCC',
                        }
                    ),
                ]),
 
                # FCBB Logo box
                html.Div(
                    style={
                        'borderRadius': '4px',
                        'padding': '8px 12px'
                    },
                    children=[
                        html.Img(src='/assets/P-Logo-FCBB.png', style={'height': '60px'}),
                        
                    ]
                ),
            ]
        ),

        # BOX 2 — Filters row
        html.Div(
            style=STYLE_DROPDOWN_ROW,
            children=[

                html.Div([
                    html.Label( style=STYLE_DROPDOWN_LABEL),
                    dcc.Dropdown(
                        options=
                            [{'label':'All Years','value':'ALL'}] +
                            [{'label':str(y),'value':y}
                             for y in sorted(df['Period'].dt.year.unique())],
                        id='year-dropdown',
                        value=['ALL'],
                        multi=True,
                        style={
                        'border':    'none',
                        'fontSize':  '13px',    # smaller text = skinnier feel
                        },
                        className='slim-dropdown'

                    ),
                ], style=STYLE_DROPDOWN_CHILD),

                html.Div([
                    html.Label( style=STYLE_DROPDOWN_LABEL),
                    dcc.Dropdown(
                        options=
                            [{'label':'All Commodities','value':'ALL'}] +
                            [{'label':str(m),'value':m}
                             for m in sorted(df['Commodity'].unique())],
                        id='commodity-dropdown',
                        value=['ALL'],
                        multi=True,
                        style={
                        'border':    'none',
                        'fontSize':  '13px',    # smaller text = skinnier feel
                        },
                        className='slim-dropdown'
                    ),
                ], style=STYLE_DROPDOWN_CHILD),

                html.Div([
                    html.Label(style=STYLE_DROPDOWN_LABEL),
                    dcc.Dropdown(
                        options=
                            [{'label':'All Provinces','value':'ALL'}] +
                            [{'label':str(c),'value':c}
                             for c in sorted(df['Province'].unique())],
                        id='province-dropdown',
                        value=['ALL'],
                        multi=True,
                        style={
                        'border':    'none',
                        'fontSize':  '13px',    # smaller text = skinnier feel
                        },
                        className='slim-dropdown'
                    ),
                ], style=STYLE_DROPDOWN_CHILD),

                html.Div([
                    html.Label(style=STYLE_DROPDOWN_LABEL),
                    dcc.Dropdown(
                        options=
                            [{'label':'All Countries','value':'ALL'}] +
                            [{'label':str(c),'value':c}
                             for c in sorted(df['Country'].unique())],
                        id='country-dropdown',
                        value=['ALL'],
                        multi=True,
                        style={
                        'border':    'none',
                        'fontSize':  '13px',    # smaller text = skinnier feel
                        },
                        className='slim-dropdown'
                    ),
                ], style=STYLE_DROPDOWN_CHILD),

            ]
        ),

        html.Div(children=[
            html.H2('Overview',
                style={
                            'margin': '20px',
                            'fontFamily': FONT_MAIN,
                            'fontSize': '28px',
                            'fontWeight': 'bold',
                            'color': BLUE_ACCENT,
                            'letterSpacing': '0.5px'}),
            html.P('Total trade flow, balance and year-over-year trend',
                   style={'margin': '6px 6px 4px 22px',
                            'fontStyle': 'italic',
                            'fontSize': '16px'
                            })

    ]),


        # BOX 3 — KPI cards
        html.Div(
            style=KPI_STYLE_ROW,
            children=[
                # Layout — static parts stay here, callback only owns the inner div
                html.Div(style=KPI_STYLE_BOX, children=[
                    html.Div(id='total-export'),   
                    html.P("Total Canada Export Value", style={'color': TEXT_GRAY, 'fontSize': '16px'})
                ]),

                html.Div(style=KPI_STYLE_BOX, children=[
                    html.Div(id='total-import'),   
                    html.P("Total Canada Import Value", style={'color': TEXT_GRAY, 'fontSize': '16px'})
                ]),

                html.Div(style=KPI_STYLE_BOX, children=[
                    html.Div(id='trade-balance'),   # ← callback only updates this part
                    html.P("Exports minus imports", style={'color': TEXT_GRAY, 'fontSize': '16px'})
                ]),
            ]
        ),

        # BOX 4 — Row 1
        html.Div([
            html.Div(dcc.Graph(id='monthly-trade'), style=STYLE_CHART_ITEM),
            html.Div(dcc.Graph(id='monthly-trade2'),   style=STYLE_CHART_ITEM),
        ], style=STYLE_CHART_ROW),

        # # BOX 5 — Row 2: Bar + Scatter
        # html.Div([
        #     html.Div(dcc.Graph(id='profit-bar'),          style=STYLE_CHART_ITEM),
        #     html.Div(dcc.Graph(id='sales-profit-scatter'), style=STYLE_CHART_ITEM),
        # ], style=STYLE_CHART_ROW),

        # # BOX 5 — Row 3: Map

        # html.Div([
        #     html.Div(dcc.Graph(id='profit-region'),          style=STYLE_CHART_ITEM),
        # ], style=STYLE_CHART_ROW),

    ]
)
@app.callback(
    Output('year-dropdown',      'value'),
    Output('commodity-dropdown', 'value'),
    Output('province-dropdown',  'value'),
    Output('country-dropdown',   'value'),
    Input('year-dropdown',       'value'),
    Input('commodity-dropdown',  'value'),
    Input('province-dropdown',   'value'),
    Input('country-dropdown',    'value'),
    prevent_initial_call=True
)
def enforce_all(year, commodity, province, country):
    def fix(selected):
        if not selected:                                   
            return ['ALL']
        if len(selected) > 1 and 'ALL' in selected:      
            return [v for v in selected if v != 'ALL']
        return selected
 
    return fix(year), fix(commodity), fix(province), fix(country)


# ── Callback 1 — KPI metrics ───────────────────────────────────────────────────
@app.callback(
    [Output('total-export',  'children'),
     Output('total-import', 'children'),
     Output('trade-balance', 'children')],
    [Input('year-dropdown',     'value'),
     Input('commodity-dropdown',   'value'),
     Input('province-dropdown', 'value'),
     Input('country-dropdown', 'value')]
)
def update_metrics(selected_year, selected_commodity, selected_province, selected_country):

    # Filter

    if not selected_year or 'ALL' in selected_year:
        filtered_df = df_kpi.copy()
    else:
        filtered_df = df[df['Period'].dt.year.isin(selected_year)]
        

    if selected_commodity and 'ALL' not in selected_commodity:
        filtered_df = filtered_df[filtered_df['Commodity'].isin(selected_commodity)]

    if selected_province and 'ALL' not in selected_province:
        filtered_df = filtered_df[filtered_df['Province'].isin(selected_province)]

    if selected_country and 'ALL' not in selected_country:
        filtered_df = filtered_df[filtered_df['Country'].isin(selected_country)]

    # Calculate
    total_export  = filtered_df[filtered_df['trade_type']=='Export']
    total_export = total_export['Value ($)'].sum()

    total_import = filtered_df[filtered_df['trade_type']=='Import']
    total_import = total_import['Value ($)'].sum()

    trade_balance = total_export-total_import

    kpi_export = [
        html.P('Total Exports',        style=KPI_STYLE_LABEL),
        html.H2(fmt_value(total_export),  style=KPI_STYLE_VALUE)
    ]
    kpi_import = [
        html.P('Total Imports',       style=KPI_STYLE_LABEL),
        html.H2(fmt_value(total_import), style=KPI_STYLE_VALUE)
    ]

    balance_color = '#C00000' if trade_balance < 0 else '#1F4E79'
    kpi_trade_balance = [
        html.P('Trade Balance', style=KPI_STYLE_LABEL),
        html.H2(fmt_value(trade_balance), style={**KPI_STYLE_VALUE, 'color': balance_color})
]

    return kpi_export, kpi_import, kpi_trade_balance



# ── Callback 2 — Charts ────────────────────────────────────────────────────────

@app.callback(
    [Output('monthly-trade','figure'),
     Output('monthly-trade2', 'figure')],


    [Input('year-dropdown',     'value'),
     Input('commodity-dropdown','value'),
     Input('province-dropdown', 'value'),
     Input('country-dropdown', 'value')]
)
def update_charts(selected_year, selected_commodity, selected_province, selected_country):

    # Filter
    if not selected_year or 'ALL' in selected_year:
        filtered_df = df.copy()
    else:
        filtered_df = df[df['Period'].dt.year.isin(selected_year)]
        

    if selected_commodity and 'ALL' not in selected_commodity:
        filtered_df = filtered_df[filtered_df['Commodity'].isin(selected_commodity)]

    if selected_province and 'ALL' not in selected_province:
        filtered_df = filtered_df[filtered_df['Province'].isin(selected_province)]

    if selected_country and 'ALL' not in selected_country:
        filtered_df = filtered_df[filtered_df['Country'].isin(selected_country)]

    # Chart 1 — Bar Chart
    fig_monthly_trade = build_monthly_chart(filtered_df)

    fig_monthly_trade2 = build_monthly_chart2(filtered_df)

    # Chart 2 — Line: Sales by Market and Year
    # sales_market = (filtered_df
    #                 .groupby(['CustomerMarket', 'OrderYear'])['GrossSales']
    #                 .sum()
    #                 .reset_index()
    #                 .sort_values('OrderYear'))
    # sales_market['OrderYear'] = sales_market['OrderYear'].astype(str)
    # fig_line = px.line(
    #     sales_market,
    #     x='OrderYear',
    #     y='GrossSales',
    #     color='CustomerMarket',
    #     markers=True,
    #     title='Sales by Market and Year',
    #     template='plotly_white'
    # )
    # fig_line.update_layout(yaxis_tickformat='$,.0f')

    # Chart 3 — Bar: Sales by Category and Year
    # sales_category = (filtered_df
    #                   .groupby(['ProductCategory', 'OrderYear'])['GrossSales']
    #                   .sum()
    #                   .reset_index())
    # sales_category['OrderYear'] = sales_category['OrderYear'].astype(str)
    # fig_bar = px.bar(
    #     sales_category,
    #     x='ProductCategory',
    #     y='GrossSales',
    #     color='OrderYear',
    #     barmode='group',
    #     title='Sales by Category and Year',
    #     template='plotly_white'
    # )
    # fig_bar.update_layout(
    #     xaxis_tickangle=-45,
    #     yaxis_tickformat='$,.0f'
    # )

    # Chart 4 — Scatter: Sales vs Profit
    # sales_product = (filtered_df
    #                  .groupby('ProductCategory')
    #                  .agg(
    #                      GrossSales = ('GrossSales', 'sum'),
    #                      Profit     = ('Profit',     'sum')
    #                  )
    #                  .reset_index())
    # fig_scatter = px.scatter(
    #     sales_product,
    #     x='GrossSales',
    #     y='Profit',
    #     size='GrossSales',
    #     text='ProductCategory',
    #     hover_name='ProductCategory',
    #     color='ProductCategory',
    #     title='Sales vs Profit by Category',
    #     template='plotly_white'
    # )
    # fig_scatter.update_traces(
    #     textposition='top center',
    #     textfont=dict(size=10),
    #     marker=dict(opacity=0.7)
    # )
    # fig_scatter.update_layout(
    #     xaxis_tickformat='$,.0f',
    #     yaxis_tickformat='$,.0f',
    #     showlegend=False
    #)

    # Chart 5 - Map: Profit vs Region
    # profit_map = (filtered_df
    #               .groupby('CustomerCountry')
    #               .agg(
    #                   GrossSales = ('GrossSales', 'sum'),
    #                   Profit     = ('Profit',     'sum')
    #               )
    #               .reset_index())

    # fig_map = px.choropleth(
    #     profit_map, locations='CustomerCountry',
    #     locationmode='country names',
    #     color= 'GrossSales',
    #     title = f'Sales by Country for Year {selected_year}',
    #     template='plotly_white',
    #     width=2400,          
    #     height=1000          
    #     )

    return fig_monthly_trade, fig_monthly_trade2

if __name__ == '__main__':
    app.run(debug=True, port=8050)

Address already in use
Port 8050 is in use by another program. Either identify and stop that program, or start the server with a different port.


SystemExit: 1

/Users/midorikawaguti/miniconda3/envs/python-data-analysis/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
